# Sampling across the miscibility gap: VCSGC ensemble

In this example, we will be covering the variance-constrained semi-grand canonical ensemble, where we can also sample structures across the miscibility gap.

In [ ]:
# once again, we load the CE model and starting structure
from icet.core.cluster_expansion import ClusterExpansion
from pathlib import Path
import numpy as np
from ase.build import make_supercell
from mchammer.calculators import ClusterExpansionCalculator

# First, we load the desired model
chemical_symbols = ["Cu", "Ni"]

max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)

ce = ClusterExpansion.read(struct_path / f"ce_model_{max_atom_num}.ce")


structure = make_supercell(
    ce.primitive_structure, 3 * np.array([[-1, 1, 1], [1, -1, 1], [1, 1, -1]])
)

temperature = 300
calculator = ClusterExpansionCalculator(structure, ce)

TASK:

- Similar to the previous exercise, perform the calculation for a set of $\phi$ and $T$ values. Start with $\phi$ at -2 towards low positive numbers.
- Can you see a phase transition for your system? (observe the discontinuity in the slope of the mixing energy)

Of course, for further analysis, we would need to properly detect the transition points. For guidelines how to do this, consult the official icet tutorials as detailed in this [paper](https://link.aps.org/doi/10.1103/PRXEnergy.3.042001).

In [ ]:
from mchammer.ensembles import VCSGCEnsemble

temperatures = [300]
phis = [-2.1]
num_steps = 50
# Perform the simulations for multiple temperatures and chemical potentials
for temperature in temperatures:
    for phi in phis:

        fname = struct_path / f"vsgc-T{temperature}-phi{phi:+.3f}.dc"
        # only do this if we did not compute it already
        if not fname.exists():
            mc = VCSGCEnsemble(
                structure=structure,
                calculator=calculator,
                temperature=temperature,
                dc_filename=fname,
                phis={
                    chemical_symbols[1]: phi,
                },
                kappa=200,
            )

            mc.run(number_of_trial_steps=len(structure) * num_steps)
            structure = mc.structure

In [ ]:
from mchammer import DataContainer
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

n_atoms = len(structure)

for temperature in temperatures:
    energies = []
    concentrations = []
    free_energy_derivatives = []
    for phi in phis:
        equilibration = 10 * n_atoms
        fname = struct_path / f"vsgc-T{temperature}-phi{phi:+.3f}.dc"
        dc = DataContainer.read(str(fname))
        occupations = dc.get("occupations", start=equilibration)

        energies.append(
            dc.get("potential", start=equilibration) / len(structure)
        )
        concs = []
        for occupation in occupations:
            concs.append(
                np.sum(occupation.symbols == chemical_symbols[1])
                / len(occupation)
            )
        concentrations.append(concs)

        free_energy_derivatives.append(
            dc.get_average(
                f"free_energy_derivative_{chemical_symbols[1]}",
                start=equilibration,
            )
        )
    concentrations = np.asarray(concentrations)
    mean_concentrations = concentrations.mean(axis=1)
    free_energy = cumulative_trapezoid(
        free_energy_derivatives, mean_concentrations, initial=0
    )
    mean_energies = np.asarray(energies).mean(axis=1)
    plt.figure(1)
    plt.scatter(
        concentrations.ravel(),
        np.asarray(energies).ravel(),
        s=2.5,
        label=f"{temperature} K",
        alpha=0.5,
    )
    plt.plot(
        mean_concentrations,
        mean_energies,
        "o-",
        label=f"{temperature} K",
    )
    plt.figure(2)
    plt.plot(
        mean_concentrations,
        np.asarray(free_energy_derivatives).ravel(),
        "o-",
        label=f"{temperature} K",
    )
    plt.figure(3)
    plt.plot(
        mean_concentrations,
        free_energy,
        "o-",
        label=f"{temperature} K",
    )


plt.figure(1)
plt.ylabel("energy / eV/atom")
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.legend()
plt.figure(2)
plt.ylabel(f"Free energy derivative / eV/atom")
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.legend()
plt.figure(3)
plt.xlabel(f"{chemical_symbols[1]} fraction")
plt.ylabel(f"Free energy / eV/atom")
plt.legend()